In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS (Ejecutar solo en Colab)
# ==============================================================================
# !pip install torch torchvision torchaudio
# !pip install opencv-python matplotlib
# !pip install git+https://github.com/facebookresearch/segment-anything.git
# !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
import os
import json
import numpy as np
import cv2
from pathlib import Path

In [ ]:
try:
    import torch
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    HAS_SAM = True
except ImportError:
    HAS_SAM = False

In [ ]:

# --- CAMBIO 1: Rutas adaptadas a entorno Colab /content ---
BASE_DIR = Path("/content") 
INDEX_PATH = BASE_DIR / "dataset_index.json"
MASK_DIR = BASE_DIR / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

# --- CAMBIO 2: Ruta del checkpoint en la raíz de Colab ---
SAM_CHECKPOINT = BASE_DIR / "sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"

def load_sam_pipeline():
    if not HAS_SAM:
        print("[ADVERTENCIA] No se ha detectado 'segment_anything' o 'torch'.")
        return None

    if not SAM_CHECKPOINT.exists():
        print(f"[ERROR] No se encuentra el checkpoint en {SAM_CHECKPOINT}.")
        return None

    # --- CAMBIO 3: Forzado de dispositivo CUDA para entorno de investigación ---
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("[CRÍTICO] GPU no detectada. Verifique 'Entorno de ejecución' en Colab.")
    
    print(f"Cargando Modelo SAM en dispositivo: {device.upper()}...")
    sam = sam_model_registry[MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
    sam.to(device=device)

    mask_generator = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )
    return mask_generator

def process_masks_to_polygons(sam_output):
    polygons_data = []
    sorted_masks = sorted(sam_output, key=(lambda x: x['area']), reverse=True)
    
    for mask_dict in sorted_masks:
        bool_mask = mask_dict['segmentation']
        binary_img = (bool_mask * 255).astype(np.uint8)
        
        contours, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_L1)
        
        if not contours:
            continue
            
        largest_contour = max(contours, key=cv2.contourArea)
        epsilon = 0.002 * cv2.arcLength(largest_contour, True)
        approx = cv2.approxPolyDP(largest_contour, epsilon, True)
        
        if len(approx) < 3:
            continue
            
        flat_poly = approx.reshape(-1, 2).tolist()
        
        polygons_data.append({
            "area": float(mask_dict['area']),
            "bbox": [float(x) for x in mask_dict['bbox']], # CAMBIO 4: Cast a float para serialización JSON
            "predicted_iou": float(mask_dict['predicted_iou']),
            "polygon": flat_poly
        })
        
    return polygons_data

def run_auto_prompt():
    print("----- Iniciando SAM Auto-Prompter en GPU -----")
    if not INDEX_PATH.exists():
        # Crear un index ficticio si no existe para pruebas rápidas
        print(f"Archivo {INDEX_PATH} no encontrado. Genere uno o suba sus imágenes.")
        return

    with open(INDEX_PATH, 'r') as f:
        dataset = json.load(f)

    generator = load_sam_pipeline()
    if not generator:
        return

    print("Comenzando inferencia acelerada...")
    for idx, item in enumerate(dataset):
        if item.get('processed_sam'):
            continue
            
        image_path = item['file_path']
        print(f"[{idx+1}/{len(dataset)}] Procesando: {image_path}...")
        
        image = cv2.imread(image_path)
        if image is None:
            print(f"Error al cargar imagen: {image_path}")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Inferencia en GPU
        masks = generator.generate(image)
        polygons = process_masks_to_polygons(masks)
        
        out_path = MASK_DIR / f"{item['id']}_sam.json"
        with open(out_path, 'w') as f:
            json.dump(polygons, f)
            
        item['processed_sam'] = True
        
    with open(INDEX_PATH, 'w') as f:
         json.dump(dataset, f, indent=4)
         
    print("\nProcesamiento SAM Terminado exitosamente.")

if __name__ == "__main__":
    run_auto_prompt()